# Constructing an ensemble of GEMs using medusa

In [ ]:
import medusa
import cobra
import numpy
import pickle
import pandas as pd
from pathlib import Path
from cobra.io import read_sbml_model
from cobra import manipulation
from medusa.core import Ensemble

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from gurobipy import setParam
setParam('OutputFlag', 0) # Suppress gurobi output

In [ ]:
# Load pan-Oryzae GEM
data_dir = Path("../model")
data_dir = data_dir.resolve()
model_path = data_dir / "panAsp_v2.xml"
panOryzae = read_sbml_model(str(model_path.resolve()))
type(panOryzae)

In [5]:
panOryzae

Name,Aspergillus_oryzae
Memory address,1bc2b914fb0
Number of metabolites,2152
Number of reactions,2250
Number of genes,1375
Number of groups,187
Objective expression,1.0*r2359 - 1.0*r2359_reverse_bde81
Compartments,"Peroxisome, Mitochondrion, Cytoplasm, Extracellular"


In [6]:
# Import BPGA table 
BPGA = pd.read_csv('../data/genome/BPGA2ortho_GEM_custom.csv', delimiter=";", dtype=str)

# Retain columns only for oryzae isolates and the oryzae tamplate model (as a reference)
BPGA.drop(['fumigatus','niger'], axis=1, inplace=True)
BPGA.rename(columns={'oryzae':'template'}, inplace=True)

# Many genes in the BPGA table are not used by any of the oryzae isolates
# -> remove these
model_sub = panOryzae.copy()
all_genes = [gene.id for gene in model_sub.genes]
BPGA = BPGA[BPGA['cluster'].isin(all_genes)]

# Make separate df for the isolate info and the gene info
BPGA_gene = BPGA[['cluster','present_in_n_genomes','cluster_type_manual']]
BPGA_isolates = BPGA.drop(['cluster','present_in_n_genomes','cluster_type_manual'], axis=1, inplace=False)

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp8hw1cil9.lp
Reading time = 0.02 seconds
: 2152 rows, 4500 columns, 17072 nonzeros


`remove_genes`: This function seems to remove genes as well as reactions catalyzed by these genes, but not metabolites. Indeed, subsequently running `prune_unused_reactions` does not alter the model. Removing unused metabolites can be achieved by running `prune_unused_metabolites`. In addition, it does not touch reactions without gene association. This is desired, however, it would also be useful to be able to remove such reactions if desired -> check which function does that 

In [21]:
# Takes 2min
gemList = []
for i in range(len(BPGA_isolates.columns)):
    gem_i = panOryzae.copy()
    isolate_name = BPGA_isolates.columns[i]
    genesToBeRemoved = BPGA_gene.cluster.values[BPGA_isolates[isolate_name].isna().values]
    manipulation.remove_genes(gem_i,
                              genesToBeRemoved,
                              True)
    _, _, = manipulation.prune_unused_metabolites(gem_i)
    gem_i.id = isolate_name
    gemList.append(gem_i)

# Also include the pan-model for downstream use (e.g. to gapfil from)
panOryzae.id = 'Pan_oryzae'
gemList.append(panOryzae)

c:\Users\gilis\OneDrive - Chalmers\Desktop\postdoc\Aspergillus\panAsp-GEM\code\.venv\Lib\site-packages\cobra\core\group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


In [24]:
# takes 1min30s
testEnsemble = Ensemble(list_of_models = gemList,
                        identifier = "testID",
                        name = "testName")

In [ ]:
# Pickle the ensemble and extracted base model
path = "../model/"
pickle.dump(testEnsemble, open(path+"panAsp_v2_ensemble.pickle","wb"))
# pickle.dump(gemList[0], open(path+"gem0.pickle","wb"))
# pickle.dump(gemList[1], open(path+"gem1.pickle","wb"))
# pickle.dump(gemList, open(path+"gemList.pickle","wb"))

In [26]:
len(testEnsemble.members)
len(testEnsemble.features)

158

1219